# 🔬 ResearchAI — Colab Demo (ngrok)

Run an **agentic, self-correcting RAG** research assistant entirely inside Colab and
get a **public URL** you can open on any device — perfect for a live competition demo.

**Stack:** Streamlit → FastAPI → LangChain → Ollama (Llama 3.1 / Qwen 2.5) · ChromaDB ·
BAAI BGE embeddings · hybrid BM25+dense + RRF · BGE reranker · PyMuPDF · Self-RAG · knowledge graph.

### ⏱️ Run order (top to bottom, ~5–8 min first time)
`GPU check → install → get code → configure → start Ollama+pull → start API → warm up → launch + ngrok`

> **Before you start:** `Runtime → Change runtime type → Hardware accelerator → GPU (T4)`.
> A free ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken makes the
> tunnel stable — paste it in the **Configuration** cell.

## 0 · GPU check
Confirm a GPU is attached (embeddings, reranker, and Ollama all use it).

In [ ]:
import subprocess
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout.strip())
else:
    print("⚠️ No GPU detected. Runtime → Change runtime type → GPU (T4), then rerun.")

## 1 · Install Ollama + system deps
Installs the Ollama binary (serves the quantized LLM locally, GPU-accelerated).

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## 2 · Get the ResearchAI code

Two ways — pick one by toggling `USE_GIT`:

* **Upload the zip** (default): run the cell, then choose `research-ai.zip` from your computer.
* **Clone a repo** (best for repeat demos): set `USE_GIT = True` and your `GIT_URL`.

In [ ]:
import os, io, zipfile, glob

USE_GIT = False                                   # ← set True to clone instead of upload
GIT_URL = "https://github.com/<your-username>/research-ai.git"

if USE_GIT:
    !git clone $GIT_URL research-ai
else:
    from google.colab import files
    print("Choose research-ai.zip …")
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(io.BytesIO(up[name])) as z:
        z.extractall(".")

# Enter the project folder (handles either layout)
root = "research-ai" if os.path.isdir("research-ai") else \
       next((os.path.dirname(p) for p in glob.glob("**/backend/main.py", recursive=True)), ".")
os.chdir(root)
print("📂 Working dir:", os.getcwd())
assert os.path.isfile("backend/main.py"), "Couldn't find the project — re-check the upload/clone."
print("✅ Code ready:", sorted(os.listdir()))

## 3 · Install Python dependencies
Colab already ships a CUDA build of PyTorch, so we install everything **except** torch
(and skip the optional `FlagEmbedding`) for a fast, conflict-free setup.

In [ ]:
pkgs = (
    "langchain langchain-core langchain-community langchain-ollama "
    "langchain-chroma langchain-huggingface langchain-text-splitters "
    "ollama chromadb sentence-transformers rank-bm25 PyMuPDF "
    "networkx pyvis fastapi uvicorn[standard] python-multipart "
    "pydantic pydantic-settings streamlit requests pyngrok"
)
!pip install -q {pkgs}
print("✅ Dependencies installed")

## 4 · Configuration  ⚙️
Colab **form** — pick your model and paste your ngrok token, then run to write `.env`.

* **`llama3.2:3b`** → fastest, snappiest live demo on a T4 (recommended for the stage).
* **`llama3.1:8b`** → the project default, higher quality, a bit slower.
* **`qwen2.5:7b`** → strong multilingual alternative.

In [ ]:
#@title Configuration { display-mode: "form" }
LLM_MODEL = "llama3.2:3b" #@param ["llama3.2:3b", "llama3.1:8b", "qwen2.5:7b"]
EMBED_DEVICE = "cuda" #@param ["cuda", "cpu"]
NGROK_AUTHTOKEN = "" #@param {type:"string"}
RERANK_TOP_K = 5 #@param {type:"integer"}
SELF_RAG_MAX_RETRIES = 2 #@param {type:"integer"}

env = f"""OLLAMA_BASE_URL=http://localhost:11434
LLM_MODEL={LLM_MODEL}
LLM_TEMPERATURE=0.1
LLM_NUM_CTX=8192
EMBED_MODEL=BAAI/bge-base-en-v1.5
RERANKER_MODEL=BAAI/bge-reranker-base
EMBED_DEVICE={EMBED_DEVICE}
DENSE_TOP_K=20
BM25_TOP_K=20
RERANK_TOP_K={RERANK_TOP_K}
MULTI_QUERY_N=3
SELF_RAG_MAX_RETRIES={SELF_RAG_MAX_RETRIES}
API_HOST=0.0.0.0
API_PORT=8000
BACKEND_URL=http://localhost:8000
NGROK_AUTHTOKEN={NGROK_AUTHTOKEN}
"""
open(".env", "w").write(env)
print(".env written:\n")
print(env)
if not NGROK_AUTHTOKEN:
    print("ℹ️ No ngrok token set — the tunnel may work but can be less stable/time-limited.")

## 5 · Start Ollama & pull the model
Launches `ollama serve` in the background, then downloads the model you chose.
First pull downloads a few GB — subsequent runs are cached for the session.

In [ ]:
import subprocess, time, requests, os

os.makedirs("logs", exist_ok=True)

def wait_for(url, timeout=180):
    start = time.time()
    while time.time() - start < timeout:
        try:
            requests.get(url, timeout=3); return True
        except Exception:
            time.sleep(2)
    return False

# (re)start Ollama
subprocess.run(["pkill", "-f", "ollama serve"], check=False)
time.sleep(2)
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("logs/ollama.log", "w"), stderr=subprocess.STDOUT)
assert wait_for("http://localhost:11434/api/tags"), "Ollama did not start — see logs/ollama.log"
print("✅ Ollama is up")

print(f"⬇️  Pulling {LLM_MODEL} … (first time takes a few minutes)")
subprocess.run(["ollama", "pull", LLM_MODEL], check=True)
print(f"✅ {LLM_MODEL} ready")

## 6 · Start the FastAPI backend
Runs the API on `:8000` in the background and waits for `/health`.

In [ ]:
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
time.sleep(2)
backend_proc = subprocess.Popen(
    ["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("logs/backend.log", "w"), stderr=subprocess.STDOUT)

health = None
for _ in range(90):
    try:
        health = requests.get("http://localhost:8000/health", timeout=3).json(); break
    except Exception:
        time.sleep(2)
print("✅ Backend health:", health if health else "NOT UP — check logs/backend.log")

## 7 · Warm up + end-to-end smoke test  🔥
This is the **demo-tuning** step: it ingests a tiny sample PDF and asks one question,
which forces the backend to load the **embeddings, BM25 index, BGE reranker, and LLM**
*now* — so your first question on stage returns instantly instead of cold-starting.
It also proves the whole grounded-Q&A pipeline works before you go live.

In [ ]:
import fitz, time, requests
from pathlib import Path

# tiny 1-page paper-like PDF
p = Path("sample.pdf")
doc = fitz.open(); pg = doc.new_page()
pg.insert_text((72, 72),
    "Introduction\n\nThis paper proposes a hybrid retrieval method that combines "
    "BM25 with dense BGE embeddings, fused via reciprocal rank fusion and reranked "
    "with a cross-encoder. We evaluate on the SQuAD dataset and optimize a "
    "cross-entropy loss. Exact-match accuracy improves by 4.2 points over a dense-only "
    "baseline. E = mc^2 is unrelated but tests equation detection.")
doc.save(p); doc.close()

with open(p, "rb") as f:
    ing = requests.post("http://localhost:8000/ingest",
                        files={"file": ("sample.pdf", f, "application/pdf")},
                        timeout=600).json()
print("📄 Ingested:", ing)

t = time.time()
ans = requests.post("http://localhost:8000/ask",
                    json={"question": "What datasets and loss function are used, and how much did accuracy improve?"},
                    timeout=600).json()
print(f"\n⏱️  Answered in {time.time()-t:.1f}s (this warm-up pays for itself on stage)\n")
print(ans["answer"])
v = ans.get("verification", {})
print(f"\n🔒 Grounding score: {v.get('grounding_score')}  "
      f"({v.get('supported')}/{v.get('total')} claims supported)")
print("🔁 Self-corrected:", ans.get("self_corrected"))

## 8 · Launch the UI + open the public ngrok URL  🌍
Starts Streamlit on `:8501` and opens a public tunnel. **Click the printed URL** —
that's your live demo. (Only the frontend is tunnelled; it talks to the API over
localhost, so a single free ngrok tunnel is all you need.)

In [ ]:
from pyngrok import ngrok, conf

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN

# clean any previous tunnels / streamlit
ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], check=False)
time.sleep(2)

streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "frontend/app.py",
     "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=open("logs/streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)

public_url = ngrok.connect(8501, "http").public_url
print("=" * 64)
print("🌍  OPEN YOUR DEMO:", public_url)
print("=" * 64)
print("Backend API docs (local): http://localhost:8000/docs")
print("If the page is blank, wait ~10s and refresh (Streamlit is still booting).")

## 🎬 Demo script (what to click)

1. **Upload** 2–3 papers in the sidebar — call out the page/chunk counts.
2. **Q&A tab** — ask a hard, specific question. Highlight the **grounding score**,
   the ✅/⚠️ per-claim badges, the **page citations**, and the **Self-RAG trace**.
3. **Agent tab** — type *"compare the two papers' methods"* → it **auto-routes** to the
   comparison tool. This is the "agentic" wow moment.
4. **Gaps tab** — surface concrete research gaps with evidence + suggested directions.
5. **Knowledge Graph tab** — extract and render the interactive graph.
6. Open the ngrok URL **on your phone** to prove it's genuinely live.

**One-liner:** *"A local, agentic RAG that doesn't just answer — it retrieves with hybrid
search, reranks, self-corrects, and proves every claim against the source page."*

---
### 🩺 Quick troubleshooting
| Symptom | Fix |
|---|---|
| UI says "Backend not reachable" | Re-run **cell 6**, check `logs/backend.log` |
| Blank ngrok page | Wait ~10s, refresh; check `logs/streamlit.log` |
| Very slow first answer | You skipped **cell 7** — run the warm-up |
| Ollama errors | Re-run **cell 5**; confirm GPU in **cell 0** |
| Out-of-memory | Use `llama3.2:3b` in the config, or set `EMBED_DEVICE=cpu` |

## 🧹 (Optional) Tear down
Stops all background processes and closes the tunnel — run when you're done.

In [ ]:
from pyngrok import ngrok
for name in ["streamlit_proc", "backend_proc", "ollama_proc"]:
    try:
        globals()[name].terminate()
    except Exception:
        pass
ngrok.kill()
for pat in ["streamlit", "uvicorn", "ollama serve"]:
    subprocess.run(["pkill", "-f", pat], check=False)
print("🛑 All services stopped.")